In [1]:
import os
os.getcwd()
# os.chdir(path)    # or you can set your working dir.

'/Users/xingfuxu/PycharmProjects/EquityPremiumPrediction3'

In [2]:
# Your working dir should include "NN_models.py", Perform_CW_test.py" and "Perform_PT_test.py" files.
from Perform_CW_test import CW_test
from Perform_PT_test import PT_test
from NN_models import Net3

In [3]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso, ElasticNet
from sklearn.cross_decomposition import PLSRegression
from sklearn.decomposition import PCA
from sklearn.preprocessing import minmax_scale
import torch
from tqdm import tqdm
#
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import AdaBoostRegressor
from xgboost import XGBRegressor
#
from sklearn.metrics import mean_squared_error

In [4]:
# set seed
torch.manual_seed(1)
np.random.seed(1)

# read quarterly data
predictor_raw = pd.read_csv("QuarterlyPredictorData1926-2020.csv")
predictor_raw.tail()

,quarter,Index,D12,E12,b/m,tbl,AAA,BAA,lty,cay,...,MA_2_9,MA_2_12,MA_3_9,MA_3_12,MOM_1,MOM_2,MOM_3,MOM_6,MOM_9,MOM_12
372,20194,3230.78,58.2406,139.47,0.229944,0.0154,0.0301,0.0388,0.0186,-0.033609,...,1,1,1,1,1,1,1,1,1,1
373,20201,2584.59,59.5806,116.33,0.306097,0.0029,0.0302,0.0429,0.0087,-0.050141,...,1,1,1,1,0,0,0,0,0,1
374,20202,3100.29,59.6834,99.23,0.259900,0.0016,0.0244,0.0364,0.0073,-0.272772,...,0,1,1,1,1,0,1,1,1,1
375,20203,3363.00,58.8512,98.22,0.241482,0.0011,0.0231,0.0336,0.0068,-0.149461,...,1,1,1,1,1,1,1,1,1,1
376,20204,3756.07,58.2788,94.13,0.219195,0.0009,0.0226,0.0316,0.0093,-0.123143,...,1,1,1,1,1,1,1,1,1,1


In [5]:
## Generating equity risk premium, 1927:01-2020:12
n_rows = predictor_raw.shape[0]
market_return = predictor_raw['CRSP_SPvw'][1:].values
risk_free_lag = predictor_raw['Rfree'][0:(n_rows - 1)].values
log_equity_premium = np.log(1 + market_return) - np.log(1 + risk_free_lag)
equity_premium = market_return - risk_free_lag

In [6]:
### Generating 12 macroeconomic variables, 1927:01-2020:12
# Notes: we exclude the log dividend-earnings ratio (DE) and the long-term yield (LTY).

# (1) Dividend-price ratio (log), DP
D12 = predictor_raw['D12'][1:].values
SP500 = predictor_raw['Index'][1:].values
DP = np.log(D12) - np.log(SP500)

# (2) Dividend yield (log), DY
SP500_lag = predictor_raw['Index'][0:(n_rows - 1)].values
DY = np.log(D12) - np.log(SP500_lag)

# (3) Earnings-price ratio (log), EP
E12 = predictor_raw['E12'][1:].values
EP = np.log(E12) - np.log(SP500)

# (4) stock variance, SVAR
SVAR = predictor_raw['svar'][1:].values

# (5) Book-to-market ratio, BM
BM = predictor_raw['b/m'][1:].values

# (6) Net equity expansion, NTIS
NTIS = predictor_raw['ntis'][1:].values

# (7) Treasury bill rate (annual %), TBL
TBL = predictor_raw['tbl'][1:].values
TBL = 100 * TBL

# (8) Long-term return (%), LTR
LTR = predictor_raw['ltr'][1:].values
LTR = 100 * LTR

# (9) Term spread (annual %), TMS
LTY = predictor_raw['lty'][1:].values
LTY = 100 * LTY
TMS = LTY - TBL

# (10) Default yield spread, DFY
AAA = predictor_raw['AAA'][1:].values
BAA = predictor_raw['BAA'][1:].values
DFY = 100 * (BAA - AAA)

# (11) Default return spread, DFR
CORPR = predictor_raw['corpr'][1:].values
DFR = 100 * CORPR - LTR

# (12) Inflation (%, lagged), INFL
INFL = predictor_raw['infl'][0:(n_rows - 1)].values
INFL = 100 * INFL

In [7]:
## Collect 12 macroeconomic variables
macro = np.concatenate([DP.reshape(-1, 1), DY.reshape(-1, 1), EP.reshape(-1, 1),
                        SVAR.reshape(-1, 1), BM.reshape(-1, 1), NTIS.reshape(-1, 1),
                        TBL.reshape(-1, 1), LTR.reshape(-1, 1), TMS.reshape(-1, 1),
                        DFY.reshape(-1, 1), DFR.reshape(-1, 1), INFL.reshape(-1, 1)], axis=1)
macro.shape

(376, 12)

In [8]:
## Collect 12 technical indicators
technical = predictor_raw[['MA_1_9', 'MA_1_12', 'MA_2_9', 'MA_2_12', 'MA_3_9',
                           'MA_3_12', 'MOM_1', 'MOM_2', 'MOM_3', 'MOM_6', 'MOM_9',
                           'MOM_12']].values[1:]
technical.shape

(376, 12)

In [9]:
predictor_matrix = np.concatenate([predictor_raw['quarter'][1:].values.reshape(-1, 1), log_equity_premium.reshape(-1, 1),
                     equity_premium.reshape(-1, 1), macro, technical], axis=1)
#
predictor_df = pd.DataFrame(predictor_matrix,
                                columns=['quarter', 'log_equity_premium', 'equity_premium', 'DP', 'DY', 'EP', 'SVAR',
                                         'BM', 'NTIS', 'TBL', 'LTR', 'TMS', 'DFY', 'DFR', 'INFL','MA_1_9', 'MA_1_12',
                                         'MA_2_9', 'MA_2_12', 'MA_3_9', 'MA_3_12', 'MOM_1', 'MOM_2', 'MOM_3', 'MOM_6',
                                         'MOM_9', 'MOM_12'])
predictor_df.head()

,quarter,log_equity_premium,equity_premium,DP,DY,EP,SVAR,BM,NTIS,TBL,...,MA_2_9,MA_2_12,MA_3_9,MA_3_12,MOM_1,MOM_2,MOM_3,MOM_6,MOM_9,MOM_12
0,19271.0,0.040386,0.041565,-2.976535,-2.944439,-2.445079,0.001681,0.469765,0.046357,3.20,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
1,19272.0,0.045197,0.046589,-3.007309,-2.948756,-2.531330,0.001819,0.452385,0.058822,3.07,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
2,19273.0,0.157540,0.171993,-3.129097,-2.980280,-2.707759,0.002826,0.380586,0.094613,2.68,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
3,19274.0,0.031700,0.032455,-3.132667,-3.102780,-2.766942,0.003093,0.374689,0.076471,3.17,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
4,19281.0,0.074012,0.077334,-3.186980,-3.107025,-2.788289,0.002933,0.363255,0.054364,3.27,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


In [10]:
predictor_df.iloc[:, 3:15] = minmax_scale(predictor_df.iloc[:, 3:15])
predictor0 = predictor_df.drop(['quarter', 'equity_premium'], axis=1)

# set the log equity premium 1-quarter ahead
predictor = np.concatenate([predictor0['log_equity_premium'][1:].values.reshape(-1, 1),
                            predictor0.iloc[0:(predictor0.shape[0] - 1), 1:]], axis=1)
N = predictor.shape[0]
n_cols = predictor.shape[1]

In [11]:
# Actual one-quarter ahead log equity premium
actual = predictor[:, [0]]

# Historical average forecasting
y_pred_HA = predictor0['log_equity_premium'].values[0:(predictor0.shape[0] - 1), ].cumsum() / np.arange(1, N + 1)
y_pred_HA = y_pred_HA.reshape(-1, 1)

## Out-of-sample: 1957:Q1-2020:Q4
in_out_1957 = predictor_df.index[predictor_df['quarter'] == 19571][0]
actual_1957 = actual[in_out_1957:, ]
y_pred_HA_1957 = y_pred_HA[in_out_1957:, ]
MSFE_HA_1957 = mean_squared_error(y_pred_HA_1957, actual_1957)

# Machine Learning methods used in GKX (2020)
y_pred_OLS_1957, y_pred_PLS_1957, y_pred_PCR_1957,  y_pred_LASSO_1957 = [], [], [], []
y_pred_ENet_1957, y_pred_GBRT_1957, y_pred_RF_1957, y_pred_NN3_1957 = [], [], [], []

## Other commonly used machine learning method
y_pred_SVR_1957, y_pred_KNR_1957, y_pred_AdaBoost_1957, y_pred_XGBoost_1957 = [], [], [], []
y_pred_combination_1957 = []

In [12]:
year_index = 1   # control the update of models each year

In [13]:
for t in tqdm(range(in_out_1957, N)):
    #
    X_train_all = predictor[:t, 1:n_cols]
    y_train_all = predictor[:t, 0]
    #
    X_train = X_train_all[0:int(len(X_train_all) * 0.85), :]
    X_validation = X_train_all[int(len(X_train_all) * 0.85):t, :]
    y_train = y_train_all[0:int(len(X_train_all) * 0.85)]
    y_validation = y_train_all[int(len(X_train_all) * 0.85):t]
    #
    if year_index % 4 == 1:
        year_index += 1
        # OLS
        OLS = LinearRegression()
        OLS.fit(X_train_all, y_train_all)
        y_pred_OLS_1957.append(OLS.predict(predictor[[t], 1:n_cols]))

        # PLS
        PLS_param = [1, 2, 3, 4, 5, 6, 7, 8]
        PLS_result = {}
        for k in PLS_param:
            PLS = PLSRegression(n_components=k)
            PLS.fit(X_train, y_train)
            mse = mean_squared_error(PLS.predict(X_validation), y_validation)
            PLS_result[mse] = k
        PLS_best_param = PLS_result[min(PLS_result.keys())]
        PLS_model = PLSRegression(n_components=PLS_best_param)
        PLS_model.fit(X_train_all, y_train_all)
        y_pred_PLS_1957.append(PLS_model.predict(predictor[[t], 1:n_cols]))

        # PCR
        PCR_param = [1, 2, 3, 4, 5, 6, 7, 8]
        PCR_result = {}
        for k in PCR_param:
            pca = PCA(n_components=k)
            pca.fit(X_train)
            comps = pca.transform(X_train)
            forecast = LinearRegression()
            forecast.fit(comps, y_train)
            mse = mean_squared_error(forecast.predict(pca.transform(X_validation)), y_validation)
            PCR_result[mse] = k
        #
        PCR_best_param = PCR_result[min(PCR_result.keys())]
        PCR_model = PCA(n_components=PCR_best_param)
        PCR_model.fit(X_train_all)
        PCR_comps = PCR_model.transform(X_train_all)
        PCR_forecast = LinearRegression()
        PCR_forecast.fit(PCR_comps, y_train_all)
        y_pred_PCR_1957.append(PCR_forecast.predict(PCR_model.transform(predictor[[t], 1:n_cols])))

        # LASSO
        LASSO_param = 10 ** np.arange(-4, -1 + 0.001, 0.1)
        LASSO_result = {}
        for alpha in LASSO_param:
            LASSO = Lasso(alpha=alpha)
            LASSO.fit(X_train, y_train)
            mse = mean_squared_error(LASSO.predict(X_validation), y_validation)
            LASSO_result[mse] = alpha
        LASSO_best_param = LASSO_result[min(LASSO_result.keys())]
        LASSO_model = Lasso(alpha=LASSO_best_param)
        LASSO_model.fit(X_train_all, y_train_all)
        y_pred_LASSO_1957.append(LASSO_model.predict(predictor[[t], 1:n_cols]))

        # ENet
        ENet_param = 10 ** np.arange(-4, -1 + 0.001, 0.1)
        ENet_result = {}
        for alpha in ENet_param:
            ENet = ElasticNet(alpha=alpha, l1_ratio=0.5)
            ENet.fit(X_train, y_train)
            mse = mean_squared_error(ENet.predict(X_validation), y_validation)
            ENet_result[mse] = alpha
        ENet_best_param = ENet_result[min(ENet_result.keys())]
        ENet_model = ElasticNet(alpha=ENet_best_param, l1_ratio=0.5)
        ENet_model.fit(X_train_all, y_train_all)
        y_pred_ENet_1957.append(ENet_model.predict(predictor[[t], 1:n_cols]))

        # GBRT
        GBRT_param = [1, 2, 3, 4, 5, 6, 7, 8]
        GBRT_result = {}
        for depth in GBRT_param:
            GBRT = GradientBoostingRegressor(max_depth=depth)
            GBRT.fit(X_train, y_train)
            mse = mean_squared_error(GBRT.predict(X_validation), y_validation)
            GBRT_result[mse] = depth
        GBRT_best_param = GBRT_result[min(GBRT_result.keys())]
        GBRT_model = GradientBoostingRegressor(max_depth=GBRT_best_param)
        GBRT_model.fit(X_train_all, y_train_all)
        y_pred_GBRT_1957.append(GBRT_model.predict(predictor[[t], 1:n_cols]))

        # RF
        RF_param =[1, 2, 3, 4, 5, 6, 7, 8]
        RF_result = {}
        for depth in RF_param:
            RF = RandomForestRegressor(max_depth=depth)
            RF.fit(X_train, y_train)
            mse = mean_squared_error(RF.predict(X_validation), y_validation)
            RF_result[mse] = depth
        RF_best_param = RF_result[min(RF_result.keys())]
        RF_model = RandomForestRegressor(max_depth=RF_best_param)
        RF_model.fit(X_train_all, y_train_all)
        y_pred_RF_1957.append(RF_model.predict(predictor[[t], 1:n_cols]))


        # NN3
        X_train_tensor = torch.tensor(X_train, dtype=torch.float)
        y_train_tensor = torch.tensor(y_train.reshape(-1, 1), dtype=torch.float)
        X_validation_tensor = torch.tensor(X_validation, dtype=torch.float)
        y_validation_tensor = torch.tensor(y_validation, dtype=torch.float)
        NN3_l2_param = 10 ** np.arange(-5, -3 + 0.0001, 0.1)
        NN3_result = {}
        NN3 = Net3(n_cols - 1, 32, 16, 8, 1)

        for l2 in NN3_l2_param:
            optimizer = torch.optim.SGD(NN3.parameters(), lr=0.01, weight_decay=l2)
            loss_func = torch.nn.MSELoss()
            for i in range(100):
                out = NN3(X_train_tensor)
                loss = loss_func(out, y_train_tensor)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            mse = mean_squared_error(NN3(X_validation_tensor).detach().numpy(), y_validation)
            NN3_result[mse] = l2
        NN3_best_param = NN3_result[min(NN3_result.keys())]
        NN3_optimizer = torch.optim.SGD(NN3.parameters(), lr=0.01, weight_decay=NN3_best_param)
        NN3_loss_func = torch.nn.MSELoss()
        X_train_all_tensor = torch.tensor(X_train_all, dtype=torch.float)
        y_train_all_tensor = torch.tensor(y_train_all.reshape(-1, 1), dtype=torch.float)
        for i in range(100):
            NN3_out = NN3(X_train_all_tensor)
            NN3_loss = NN3_loss_func(NN3_out, y_train_all_tensor)
            NN3_optimizer.zero_grad()
            NN3_loss.backward()
            NN3_optimizer.step()
        y_pred_NN3_1957.append(NN3(torch.tensor(predictor[[t], 1:n_cols],
                                                dtype=torch.float)).detach().numpy()[0])
        ## Other commmonly used ML methods
        # SVR
        SVR_param = ['linear', 'poly', 'rbf', 'sigmoid']
        SVR_result = {}
        for kernel in SVR_param:
            SVR_tmp = SVR(kernel=kernel)
            SVR_tmp.fit(X_train, y_train)
            mse = mean_squared_error(SVR_tmp.predict(X_validation), y_validation)
            SVR_result[mse] = kernel
        SVR_best_param = SVR_result[min(SVR_result.keys())]
        SVR_model = SVR(kernel=SVR_best_param)
        SVR_model.fit(X_train_all, y_train_all)
        y_pred_SVR_1957.append(SVR_model.predict(predictor[[t], 1:n_cols]))

        # KNR
        KNR = KNeighborsRegressor()
        KNR_param = [5, 10, 20, 25, 30, 40, 50, 60, 70]
        KNR_result = {}
        for n_neighbors in KNR_param:
            KNR = KNeighborsRegressor(n_neighbors=n_neighbors)
            KNR.fit(X_train, y_train)
            mse = mean_squared_error(KNR.predict(X_validation), y_validation)
            KNR_result[mse] = n_neighbors
        KNR_best_param = KNR_result[min(KNR_result.keys())]
        KNR_model = KNeighborsRegressor(n_neighbors=KNR_best_param)
        KNR_model.fit(X_train_all, y_train_all)
        y_pred_KNR_1957.append(KNR_model.predict(predictor[[t], 1:n_cols]))

        # AdaBoost
        AdaBoost_param = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
        AdaBoost_result = {}
        for n_estimators in AdaBoost_param:
            AdaBoost = AdaBoostRegressor(n_estimators=n_estimators)
            AdaBoost.fit(X_train, y_train)
            mse = mean_squared_error(AdaBoost.predict(X_validation), y_validation)
            AdaBoost_result[mse] = n_estimators
        AdaBoost_best_param = AdaBoost_result[min(AdaBoost_result.keys())]
        AdaBoost_model = AdaBoostRegressor(n_estimators=AdaBoost_best_param)
        AdaBoost_model.fit(X_train_all, y_train_all)
        y_pred_AdaBoost_1957.append(AdaBoost_model.predict(predictor[[t], 1:n_cols]))

        # XGBoost
        XGBoost_param = [1, 2, 3, 4, 5, 6, 7, 8]
        XGBoost_result = {}
        for max_depth in XGBoost_param:
            XGBoost = XGBRegressor(max_depth=max_depth)
            XGBoost.fit(X_train, y_train)
            mse = mean_squared_error(XGBoost.predict(X_validation), y_validation)
            XGBoost_result[mse] = max_depth
        XGB_best_param = XGBoost_result[min(XGBoost_result.keys())]
        XGB_model = XGBRegressor(max_depth=XGB_best_param)
        XGB_model.fit(X_train_all, y_train_all)
        y_pred_XGBoost_1957.append(XGB_model.predict(predictor[[t], 1:n_cols]))
    else:
        year_index += 1
        y_pred_OLS_1957.append(OLS.predict(predictor[[t], 1:n_cols]))
        y_pred_PLS_1957.append(PLS_model.predict(predictor[[t], 1:n_cols]))
        y_pred_PCR_1957.append(PCR_forecast.predict(PCR_model.transform(predictor[[t], 1:n_cols])))
        y_pred_LASSO_1957.append(LASSO_model.predict(predictor[[t], 1:n_cols]))
        y_pred_ENet_1957.append(ENet_model.predict(predictor[[t], 1:n_cols]))
        y_pred_GBRT_1957.append(GBRT_model.predict(predictor[[t], 1:n_cols]))
        y_pred_RF_1957.append(RF_model.predict(predictor[[t], 1:n_cols]))
        y_pred_NN3_1957.append(NN3(torch.tensor(predictor[[t], 1:n_cols], dtype=torch.float)).detach().numpy()[0])
        # Other commmonly used ML methods
        y_pred_SVR_1957.append(SVR_model.predict(predictor[[t], 1:n_cols]))
        y_pred_KNR_1957.append(KNR_model.predict(predictor[[t], 1:n_cols]))
        y_pred_AdaBoost_1957.append(AdaBoost_model.predict(predictor[[t], 1:n_cols]))
        y_pred_XGBoost_1957.append(XGB_model.predict(predictor[[t], 1:n_cols]))


# Performance compared with HA
# OLS
y_pred_OLS_1957 = np.array(y_pred_OLS_1957).reshape(-1, 1)
MSFE_OLS_1957 = mean_squared_error(y_pred_OLS_1957, actual_1957)
OOS_R_OLS_1957 = 1 - MSFE_OLS_1957 / MSFE_HA_1957
MSFE_adjusted_OLS_1957, p_OLS_1957 = CW_test(actual_1957, y_pred_HA_1957, y_pred_OLS_1957)
success_ratio_OLS_1957, PT_OLS_1957, p2_OLS_1957 = PT_test(actual_1957, y_pred_OLS_1957)
# PLS
y_pred_PLS_1957 = np.array(y_pred_PLS_1957).reshape(-1, 1)
MSFE_PLS_1957 = mean_squared_error(y_pred_PLS_1957, actual_1957)
OOS_R_PLS_1957 = 1 - MSFE_PLS_1957 / MSFE_HA_1957
MSFE_adjusted_PLS_1957, p_PLS_1957 = CW_test(actual_1957, y_pred_HA_1957, y_pred_PLS_1957)
success_ratio_PLS_1957, PT_PLS_1957, p2_PLS_1957 = PT_test(actual_1957, y_pred_PLS_1957)
# PCR
y_pred_PCR_1957 = np.array(y_pred_PCR_1957).reshape(-1, 1)
MSFE_PCR_1957 = mean_squared_error(y_pred_PCR_1957, actual_1957)
OOS_R_PCR_1957 = 1 - MSFE_PCR_1957 / MSFE_HA_1957
MSFE_adjusted_PCR_1957, p_PCR_1957 = CW_test(actual_1957, y_pred_HA_1957, y_pred_PCR_1957)
success_ratio_PCR_1957, PT_PCR_1957, p2_PCR_1957 = PT_test(actual_1957, y_pred_PCR_1957)
# LASSO
y_pred_LASSO_1957 = np.array(y_pred_LASSO_1957).reshape(-1, 1)
MSFE_LASSO_1957 = mean_squared_error(y_pred_LASSO_1957, actual_1957)
OOS_R_LASSO_1957 = 1 - MSFE_LASSO_1957 / MSFE_HA_1957
MSFE_adjusted_LASSO_1957, p_LASSO_1957 = CW_test(actual_1957, y_pred_HA_1957, y_pred_LASSO_1957)
success_ratio_LASSO_1957, PT_LASSO_1957, p2_LASSO_1957 = PT_test(actual_1957, y_pred_LASSO_1957)
# ENet
y_pred_ENet_1957 = np.array(y_pred_ENet_1957).reshape(-1, 1)
MSFE_ENet_1957 = mean_squared_error(y_pred_ENet_1957, actual_1957)
OOS_R_ENet_1957 = 1 - MSFE_ENet_1957 / MSFE_HA_1957
MSFE_adjusted_ENet_1957, p_ENet_1957 = CW_test(actual_1957, y_pred_HA_1957, y_pred_ENet_1957)
success_ratio_ENet_1957, PT_ENet_1957, p2_ENet_1957 = PT_test(actual_1957, y_pred_ENet_1957)
# GBRT
y_pred_GBRT_1957 = np.array(y_pred_GBRT_1957).reshape(-1, 1)
MSFE_GBRT_1957 = mean_squared_error(y_pred_GBRT_1957, actual_1957)
OOS_R_GBRT_1957 = 1 - MSFE_GBRT_1957 / MSFE_HA_1957
MSFE_adjusted_GBRT_1957, p_GBRT_1957 = CW_test(actual_1957, y_pred_HA_1957, y_pred_GBRT_1957)
success_ratio_GBRT_1957, PT_GBRT_1957, p2_GBRT_1957 = PT_test(actual_1957, y_pred_GBRT_1957)
# RF
y_pred_RF_1957 = np.array(y_pred_RF_1957).reshape(-1, 1)
MSFE_RF_1957 = mean_squared_error(y_pred_RF_1957, actual_1957)
OOS_R_RF_1957 = 1 - MSFE_RF_1957 / MSFE_HA_1957
MSFE_adjusted_RF_1957, p_RF_1957 = CW_test(actual_1957, y_pred_HA_1957, y_pred_RF_1957)
success_ratio_RF_1957, PT_RF_1957, p2_RF_1957 = PT_test(actual_1957, y_pred_RF_1957)
# NN3
y_pred_NN3_1957 = np.array(y_pred_NN3_1957).reshape(-1, 1)
MSFE_NN3_1957 = mean_squared_error(y_pred_NN3_1957, actual_1957)
OOS_R_NN3_1957 = 1 - MSFE_NN3_1957 / MSFE_HA_1957
MSFE_adjusted_NN3_1957, p_NN3_1957 = CW_test(actual_1957, y_pred_HA_1957, y_pred_NN3_1957)
success_ratio_NN3_1957, PT_NN3_1957, p2_NN3_1957 = PT_test(actual_1957, y_pred_NN3_1957)
# SVR
y_pred_SVR_1957 = np.array(y_pred_SVR_1957).reshape(-1, 1)
MSFE_SVR_1957 = mean_squared_error(y_pred_SVR_1957, actual_1957)
OOS_R_SVR_1957 = 1 - MSFE_SVR_1957 / MSFE_HA_1957
MSFE_adjusted_SVR_1957, p_SVR_1957 = CW_test(actual_1957, y_pred_HA_1957, y_pred_SVR_1957)
success_ratio_SVR_1957, PT_SVR_1957, p2_SVR_1957 = PT_test(actual_1957, y_pred_SVR_1957)
# KNR
y_pred_KNR_1957 = np.array(y_pred_KNR_1957).reshape(-1, 1)
MSFE_KNR_1957 = mean_squared_error(y_pred_KNR_1957, actual_1957)
OOS_R_KNR_1957 = 1 - MSFE_KNR_1957 / MSFE_HA_1957
MSFE_adjusted_KNR_1957, p_KNR_1957 = CW_test(actual_1957, y_pred_HA_1957, y_pred_KNR_1957)
success_ratio_KNR_1957, PT_KNR_1957, p2_KNR_1957 = PT_test(actual_1957, y_pred_KNR_1957)
# AdaBoost
y_pred_AdaBoost_1957 = np.array(y_pred_AdaBoost_1957).reshape(-1, 1)
MSFE_AdaBoost_1957 = mean_squared_error(y_pred_AdaBoost_1957, actual_1957)
OOS_R_AdaBoost_1957 = 1 - MSFE_AdaBoost_1957 / MSFE_HA_1957
MSFE_adjusted_AdaBoost_1957, p_AdaBoost_1957 = CW_test(actual_1957, y_pred_HA_1957, y_pred_AdaBoost_1957)
success_ratio_AdaBoost_1957, PT_AdaBoost_1957, p2_AdaBoost_1957 = PT_test(actual_1957, y_pred_AdaBoost_1957)
# XGBoost
y_pred_XGBoost_1957 = np.array(y_pred_XGBoost_1957).reshape(-1, 1)
MSFE_XGBoost_1957 = mean_squared_error(y_pred_XGBoost_1957, actual_1957)
OOS_R_XGBoost_1957 = 1 - MSFE_XGBoost_1957 / MSFE_HA_1957
MSFE_adjusted_XGBoost_1957, p_XGBoost_1957 = CW_test(actual_1957, y_pred_HA_1957, y_pred_XGBoost_1957)
success_ratio_XGBoost_1957, PT_XGBoost_1957, p2_XGBoost_1957 = PT_test(actual_1957, y_pred_XGBoost_1957)
# Combination
y_pred_combination_1957 = np.concatenate([y_pred_OLS_1957, y_pred_PLS_1957, y_pred_PCR_1957, y_pred_LASSO_1957,
                                          y_pred_ENet_1957, y_pred_GBRT_1957, y_pred_RF_1957, y_pred_NN3_1957,
                                          y_pred_SVR_1957, y_pred_KNR_1957, y_pred_AdaBoost_1957,
                                          y_pred_XGBoost_1957], axis=1).mean(axis=1).reshape(-1, 1)
MSFE_combination_1957 = mean_squared_error(y_pred_combination_1957, actual_1957)
OOS_R_combination_1957 = 1 - MSFE_combination_1957 / MSFE_HA_1957
MSFE_adjusted_combination_1957, p_combination_1957 = CW_test(actual_1957, y_pred_HA_1957, y_pred_combination_1957)
success_ratio_combination_1957, PT_combination_1957, p2_combination_1957 = PT_test(actual_1957, y_pred_combination_1957)
# success ratio of HA
success_ratio_HA_1957, PT_HA_1957, p2_HA_1957 = PT_test(actual_1957, y_pred_HA_1957)

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 255/255 [04:00<00:00,  1.06it/s]
/Users/xingfuxu/PycharmProjects/EquityPremiumPrediction3/Perform_PT_test.py:36: RuntimeWarning: invalid value encountered in double_scalars
  stat = (p_hat - p_star) / np.sqrt(p_hat_var - p_star_var)


In [14]:
## Out-of-sample: 1990:Q1-2020:Q4
in_out_1990 = predictor_df.index[predictor_df['quarter'] == 19901][0]
actual_1990 = actual[in_out_1990:, ]
y_pred_HA_1990 = y_pred_HA[in_out_1990:, ]
MSFE_HA_1990 = mean_squared_error(y_pred_HA_1990, actual_1990)

# Machine Learning methods used in GKX (2020)
y_pred_OLS_1990, y_pred_PLS_1990, y_pred_PCR_1990,  y_pred_LASSO_1990 = [], [], [], []
y_pred_ENet_1990, y_pred_GBRT_1990, y_pred_RF_1990, y_pred_NN3_1990 = [], [], [], []

## Other commonly used machine learning method
y_pred_SVR_1990, y_pred_KNR_1990, y_pred_AdaBoost_1990, y_pred_XGBoost_1990 = [], [], [], []
y_pred_combination_1990 = []

year_index = 1   # control the update of model each year
for t in tqdm(range(in_out_1990, N)):
    #
    X_train_all = predictor[:t, 1:n_cols]
    y_train_all = predictor[:t, 0]
    #
    X_train = X_train_all[0:int(len(X_train_all) * 0.85), :]
    X_validation = X_train_all[int(len(X_train_all) * 0.85):t, :]
    y_train = y_train_all[0:int(len(X_train_all) * 0.85)]
    y_validation = y_train_all[int(len(X_train_all) * 0.85):t]
    #
    if year_index % 4 == 1:
        year_index += 1
        # OLS
        OLS = LinearRegression()
        OLS.fit(X_train_all, y_train_all)
        y_pred_OLS_1990.append(OLS.predict(predictor[[t], 1:n_cols]))

        # PLS
        PLS_param = [1, 2, 3, 4, 5, 6, 7, 8]
        PLS_result = {}
        for k in PLS_param:
            PLS = PLSRegression(n_components=k)
            PLS.fit(X_train, y_train)
            mse = mean_squared_error(PLS.predict(X_validation), y_validation)
            PLS_result[mse] = k
        PLS_best_param = PLS_result[min(PLS_result.keys())]
        PLS_model = PLSRegression(n_components=PLS_best_param)
        PLS_model.fit(X_train_all, y_train_all)
        y_pred_PLS_1990.append(PLS_model.predict(predictor[[t], 1:n_cols]))

        # PCR
        PCR_param = [1, 2, 3, 4, 5, 6, 7, 8]
        PCR_result = {}
        for k in PCR_param:
            pca = PCA(n_components=k)
            pca.fit(X_train)
            comps = pca.transform(X_train)
            forecast = LinearRegression()
            forecast.fit(comps, y_train)
            mse = mean_squared_error(forecast.predict(pca.transform(X_validation)), y_validation)
            PCR_result[mse] = k
        #
        PCR_best_param = PCR_result[min(PCR_result.keys())]
        PCR_model = PCA(n_components=PCR_best_param)
        PCR_model.fit(X_train_all)
        PCR_comps = PCR_model.transform(X_train_all)
        PCR_forecast = LinearRegression()
        PCR_forecast.fit(PCR_comps, y_train_all)
        y_pred_PCR_1990.append(PCR_forecast.predict(PCR_model.transform(predictor[[t], 1:n_cols])))

        # LASSO
        LASSO_param = 10 ** np.arange(-4, -1 + 0.001, 0.1)
        LASSO_result = {}
        for alpha in LASSO_param:
            LASSO = Lasso(alpha=alpha)
            LASSO.fit(X_train, y_train)
            mse = mean_squared_error(LASSO.predict(X_validation), y_validation)
            LASSO_result[mse] = alpha
        LASSO_best_param = LASSO_result[min(LASSO_result.keys())]
        LASSO_model = Lasso(alpha=LASSO_best_param)
        LASSO_model.fit(X_train_all, y_train_all)
        y_pred_LASSO_1990.append(LASSO_model.predict(predictor[[t], 1:n_cols]))

        # ENet
        ENet_param = 10 ** np.arange(-4, -1 + 0.001, 0.1)
        ENet_result = {}
        for alpha in ENet_param:
            ENet = ElasticNet(alpha=alpha, l1_ratio=0.5)
            ENet.fit(X_train, y_train)
            mse = mean_squared_error(ENet.predict(X_validation), y_validation)
            ENet_result[mse] = alpha
        ENet_best_param = ENet_result[min(ENet_result.keys())]
        ENet_model = ElasticNet(alpha=ENet_best_param, l1_ratio=0.5)
        ENet_model.fit(X_train_all, y_train_all)
        y_pred_ENet_1990.append(ENet_model.predict(predictor[[t], 1:n_cols]))

        # GBRT
        GBRT_param = [1, 2, 3, 4, 5, 6, 7, 8]
        GBRT_result = {}
        for depth in GBRT_param:
            GBRT = GradientBoostingRegressor(max_depth=depth)
            GBRT.fit(X_train, y_train)
            mse = mean_squared_error(GBRT.predict(X_validation), y_validation)
            GBRT_result[mse] = depth
        GBRT_best_param = GBRT_result[min(GBRT_result.keys())]
        GBRT_model = GradientBoostingRegressor(max_depth=GBRT_best_param)
        GBRT_model.fit(X_train_all, y_train_all)
        y_pred_GBRT_1990.append(GBRT_model.predict(predictor[[t], 1:n_cols]))

        # RF
        RF_param =[1, 2, 3, 4, 5, 6, 7, 8]
        RF_result = {}
        for depth in RF_param:
            RF = RandomForestRegressor(max_depth=depth)
            RF.fit(X_train, y_train)
            mse = mean_squared_error(RF.predict(X_validation), y_validation)
            RF_result[mse] = depth
        RF_best_param = RF_result[min(RF_result.keys())]
        RF_model = RandomForestRegressor(max_depth=RF_best_param)
        RF_model.fit(X_train_all, y_train_all)
        y_pred_RF_1990.append(RF_model.predict(predictor[[t], 1:n_cols]))


        # NN3
        X_train_tensor = torch.tensor(X_train, dtype=torch.float)
        y_train_tensor = torch.tensor(y_train.reshape(-1, 1), dtype=torch.float)
        X_validation_tensor = torch.tensor(X_validation, dtype=torch.float)
        y_validation_tensor = torch.tensor(y_validation, dtype=torch.float)
        NN3_l2_param = 10 ** np.arange(-5, -3 + 0.0001, 0.1)
        NN3_result = {}
        NN3 = Net3(n_cols - 1, 32, 16, 8, 1)

        for l2 in NN3_l2_param:
            optimizer = torch.optim.SGD(NN3.parameters(), lr=0.01, weight_decay=l2)
            loss_func = torch.nn.MSELoss()
            for i in range(100):
                out = NN3(X_train_tensor)
                loss = loss_func(out, y_train_tensor)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            mse = mean_squared_error(NN3(X_validation_tensor).detach().numpy(), y_validation)
            NN3_result[mse] = l2
        NN3_best_param = NN3_result[min(NN3_result.keys())]
        NN3_optimizer = torch.optim.SGD(NN3.parameters(), lr=0.01, weight_decay=NN3_best_param)
        NN3_loss_func = torch.nn.MSELoss()
        X_train_all_tensor = torch.tensor(X_train_all, dtype=torch.float)
        y_train_all_tensor = torch.tensor(y_train_all.reshape(-1, 1), dtype=torch.float)
        for i in range(100):
            NN3_out = NN3(X_train_all_tensor)
            NN3_loss = NN3_loss_func(NN3_out, y_train_all_tensor)
            NN3_optimizer.zero_grad()
            NN3_loss.backward()
            NN3_optimizer.step()
        y_pred_NN3_1990.append(NN3(torch.tensor(predictor[[t], 1:n_cols],
                                                dtype=torch.float)).detach().numpy()[0])
        ## Other commmonly used ML methods
        # SVR
        SVR_param = ['linear', 'poly', 'rbf', 'sigmoid']
        SVR_result = {}
        for kernel in SVR_param:
            SVR_tmp = SVR(kernel=kernel)
            SVR_tmp.fit(X_train, y_train)
            mse = mean_squared_error(SVR_tmp.predict(X_validation), y_validation)
            SVR_result[mse] = kernel
        SVR_best_param = SVR_result[min(SVR_result.keys())]
        SVR_model = SVR(kernel=SVR_best_param)
        SVR_model.fit(X_train_all, y_train_all)
        y_pred_SVR_1990.append(SVR_model.predict(predictor[[t], 1:n_cols]))

        # KNR
        KNR = KNeighborsRegressor()
        KNR_param = [5, 10, 20, 25, 30, 40, 50, 60, 70]
        KNR_result = {}
        for n_neighbors in KNR_param:
            KNR = KNeighborsRegressor(n_neighbors=n_neighbors)
            KNR.fit(X_train, y_train)
            mse = mean_squared_error(KNR.predict(X_validation), y_validation)
            KNR_result[mse] = n_neighbors
        KNR_best_param = KNR_result[min(KNR_result.keys())]
        KNR_model = KNeighborsRegressor(n_neighbors=KNR_best_param)
        KNR_model.fit(X_train_all, y_train_all)
        y_pred_KNR_1990.append(KNR_model.predict(predictor[[t], 1:n_cols]))

        # AdaBoost
        AdaBoost_param = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
        AdaBoost_result = {}
        for n_estimators in AdaBoost_param:
            AdaBoost = AdaBoostRegressor(n_estimators=n_estimators)
            AdaBoost.fit(X_train, y_train)
            mse = mean_squared_error(AdaBoost.predict(X_validation), y_validation)
            AdaBoost_result[mse] = n_estimators
        AdaBoost_best_param = AdaBoost_result[min(AdaBoost_result.keys())]
        AdaBoost_model = AdaBoostRegressor(n_estimators=AdaBoost_best_param)
        AdaBoost_model.fit(X_train_all, y_train_all)
        y_pred_AdaBoost_1990.append(AdaBoost_model.predict(predictor[[t], 1:n_cols]))

        # XGBoost
        XGBoost_param = [1, 2, 3, 4, 5, 6, 7, 8]
        XGBoost_result = {}
        for max_depth in XGBoost_param:
            XGBoost = XGBRegressor(max_depth=max_depth)
            XGBoost.fit(X_train, y_train)
            mse = mean_squared_error(XGBoost.predict(X_validation), y_validation)
            XGBoost_result[mse] = max_depth
        XGB_best_param = XGBoost_result[min(XGBoost_result.keys())]
        XGB_model = XGBRegressor(max_depth=XGB_best_param)
        XGB_model.fit(X_train_all, y_train_all)
        y_pred_XGBoost_1990.append(XGB_model.predict(predictor[[t], 1:n_cols]))
    else:
        year_index += 1
        y_pred_OLS_1990.append(OLS.predict(predictor[[t], 1:n_cols]))
        y_pred_PLS_1990.append(PLS_model.predict(predictor[[t], 1:n_cols]))
        y_pred_PCR_1990.append(PCR_forecast.predict(PCR_model.transform(predictor[[t], 1:n_cols])))
        y_pred_LASSO_1990.append(LASSO_model.predict(predictor[[t], 1:n_cols]))
        y_pred_ENet_1990.append(ENet_model.predict(predictor[[t], 1:n_cols]))
        y_pred_GBRT_1990.append(GBRT_model.predict(predictor[[t], 1:n_cols]))
        y_pred_RF_1990.append(RF_model.predict(predictor[[t], 1:n_cols]))
        y_pred_NN3_1990.append(NN3(torch.tensor(predictor[[t], 1:n_cols], dtype=torch.float)).detach().numpy()[0])
        # Other commmonly used ML methods
        y_pred_SVR_1990.append(SVR_model.predict(predictor[[t], 1:n_cols]))
        y_pred_KNR_1990.append(KNR_model.predict(predictor[[t], 1:n_cols]))
        y_pred_AdaBoost_1990.append(AdaBoost_model.predict(predictor[[t], 1:n_cols]))
        y_pred_XGBoost_1990.append(XGB_model.predict(predictor[[t], 1:n_cols]))

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 123/123 [02:11<00:00,  1.07s/it]


In [15]:

# Performance compared with HA
# OLS
y_pred_OLS_1990 = np.array(y_pred_OLS_1990).reshape(-1, 1)
MSFE_OLS_1990 = mean_squared_error(y_pred_OLS_1990, actual_1990)
OOS_R_OLS_1990 = 1 - MSFE_OLS_1990 / MSFE_HA_1990
MSFE_adjusted_OLS_1990, p_OLS_1990 = CW_test(actual_1990, y_pred_HA_1990, y_pred_OLS_1990)
success_ratio_OLS_1990, PT_OLS_1990, p2_OLS_1990 = PT_test(actual_1990, y_pred_OLS_1990)
# PLS
y_pred_PLS_1990 = np.array(y_pred_PLS_1990).reshape(-1, 1)
MSFE_PLS_1990 = mean_squared_error(y_pred_PLS_1990, actual_1990)
OOS_R_PLS_1990 = 1 - MSFE_PLS_1990 / MSFE_HA_1990
MSFE_adjusted_PLS_1990, p_PLS_1990 = CW_test(actual_1990, y_pred_HA_1990, y_pred_PLS_1990)
success_ratio_PLS_1990, PT_PLS_1990, p2_PLS_1990 = PT_test(actual_1990, y_pred_PLS_1990)
# PCR
y_pred_PCR_1990 = np.array(y_pred_PCR_1990).reshape(-1, 1)
MSFE_PCR_1990 = mean_squared_error(y_pred_PCR_1990, actual_1990)
OOS_R_PCR_1990 = 1 - MSFE_PCR_1990 / MSFE_HA_1990
MSFE_adjusted_PCR_1990, p_PCR_1990 = CW_test(actual_1990, y_pred_HA_1990, y_pred_PCR_1990)
success_ratio_PCR_1990, PT_PCR_1990, p2_PCR_1990 = PT_test(actual_1990, y_pred_PCR_1990)
# LASSO
y_pred_LASSO_1990 = np.array(y_pred_LASSO_1990).reshape(-1, 1)
MSFE_LASSO_1990 = mean_squared_error(y_pred_LASSO_1990, actual_1990)
OOS_R_LASSO_1990 = 1 - MSFE_LASSO_1990 / MSFE_HA_1990
MSFE_adjusted_LASSO_1990, p_LASSO_1990 = CW_test(actual_1990, y_pred_HA_1990, y_pred_LASSO_1990)
success_ratio_LASSO_1990, PT_LASSO_1990, p2_LASSO_1990 = PT_test(actual_1990, y_pred_LASSO_1990)
# ENet
y_pred_ENet_1990 = np.array(y_pred_ENet_1990).reshape(-1, 1)
MSFE_ENet_1990 = mean_squared_error(y_pred_ENet_1990, actual_1990)
OOS_R_ENet_1990 = 1 - MSFE_ENet_1990 / MSFE_HA_1990
MSFE_adjusted_ENet_1990, p_ENet_1990 = CW_test(actual_1990, y_pred_HA_1990, y_pred_ENet_1990)
success_ratio_ENet_1990, PT_ENet_1990, p2_ENet_1990 = PT_test(actual_1990, y_pred_ENet_1990)
# GBRT
y_pred_GBRT_1990 = np.array(y_pred_GBRT_1990).reshape(-1, 1)
MSFE_GBRT_1990 = mean_squared_error(y_pred_GBRT_1990, actual_1990)
OOS_R_GBRT_1990 = 1 - MSFE_GBRT_1990 / MSFE_HA_1990
MSFE_adjusted_GBRT_1990, p_GBRT_1990 = CW_test(actual_1990, y_pred_HA_1990, y_pred_GBRT_1990)
success_ratio_GBRT_1990, PT_GBRT_1990, p2_GBRT_1990 = PT_test(actual_1990, y_pred_GBRT_1990)
# RF
y_pred_RF_1990 = np.array(y_pred_RF_1990).reshape(-1, 1)
MSFE_RF_1990 = mean_squared_error(y_pred_RF_1990, actual_1990)
OOS_R_RF_1990 = 1 - MSFE_RF_1990 / MSFE_HA_1990
MSFE_adjusted_RF_1990, p_RF_1990 = CW_test(actual_1990, y_pred_HA_1990, y_pred_RF_1990)
success_ratio_RF_1990, PT_RF_1990, p2_RF_1990 = PT_test(actual_1990, y_pred_RF_1990)
# NN3
y_pred_NN3_1990 = np.array(y_pred_NN3_1990).reshape(-1, 1)
MSFE_NN3_1990 = mean_squared_error(y_pred_NN3_1990, actual_1990)
OOS_R_NN3_1990 = 1 - MSFE_NN3_1990 / MSFE_HA_1990
MSFE_adjusted_NN3_1990, p_NN3_1990 = CW_test(actual_1990, y_pred_HA_1990, y_pred_NN3_1990)
success_ratio_NN3_1990, PT_NN3_1990, p2_NN3_1990 = PT_test(actual_1990, y_pred_NN3_1990)
# SVR
y_pred_SVR_1990 = np.array(y_pred_SVR_1990).reshape(-1, 1)
MSFE_SVR_1990 = mean_squared_error(y_pred_SVR_1990, actual_1990)
OOS_R_SVR_1990 = 1 - MSFE_SVR_1990 / MSFE_HA_1990
MSFE_adjusted_SVR_1990, p_SVR_1990 = CW_test(actual_1990, y_pred_HA_1990, y_pred_SVR_1990)
success_ratio_SVR_1990, PT_SVR_1990, p2_SVR_1990 = PT_test(actual_1990, y_pred_SVR_1990)
# KNR
y_pred_KNR_1990 = np.array(y_pred_KNR_1990).reshape(-1, 1)
MSFE_KNR_1990 = mean_squared_error(y_pred_KNR_1990, actual_1990)
OOS_R_KNR_1990 = 1 - MSFE_KNR_1990 / MSFE_HA_1990
MSFE_adjusted_KNR_1990, p_KNR_1990 = CW_test(actual_1990, y_pred_HA_1990, y_pred_KNR_1990)
success_ratio_KNR_1990, PT_KNR_1990, p2_KNR_1990 = PT_test(actual_1990, y_pred_KNR_1990)
# AdaBoost
y_pred_AdaBoost_1990 = np.array(y_pred_AdaBoost_1990).reshape(-1, 1)
MSFE_AdaBoost_1990 = mean_squared_error(y_pred_AdaBoost_1990, actual_1990)
OOS_R_AdaBoost_1990 = 1 - MSFE_AdaBoost_1990 / MSFE_HA_1990
MSFE_adjusted_AdaBoost_1990, p_AdaBoost_1990 = CW_test(actual_1990, y_pred_HA_1990, y_pred_AdaBoost_1990)
success_ratio_AdaBoost_1990, PT_AdaBoost_1990, p2_AdaBoost_1990 = PT_test(actual_1990, y_pred_AdaBoost_1990)
# XGBoost
y_pred_XGBoost_1990 = np.array(y_pred_XGBoost_1990).reshape(-1, 1)
MSFE_XGBoost_1990 = mean_squared_error(y_pred_XGBoost_1990, actual_1990)
OOS_R_XGBoost_1990 = 1 - MSFE_XGBoost_1990 / MSFE_HA_1990
MSFE_adjusted_XGBoost_1990, p_XGBoost_1990 = CW_test(actual_1990, y_pred_HA_1990, y_pred_XGBoost_1990)
success_ratio_XGBoost_1990, PT_XGBoost_1990, p2_XGBoost_1990 = PT_test(actual_1990, y_pred_XGBoost_1990)
# Combination
y_pred_combination_1990 = np.concatenate([y_pred_OLS_1990, y_pred_PLS_1990, y_pred_PCR_1990, y_pred_LASSO_1990,
                                          y_pred_ENet_1990, y_pred_GBRT_1990, y_pred_RF_1990, y_pred_NN3_1990,
                                          y_pred_SVR_1990, y_pred_KNR_1990, y_pred_AdaBoost_1990,
                                          y_pred_XGBoost_1990], axis=1).mean(axis=1).reshape(-1, 1)
MSFE_combination_1990 = mean_squared_error(y_pred_combination_1990, actual_1990)
OOS_R_combination_1990 = 1 - MSFE_combination_1990 / MSFE_HA_1990
MSFE_adjusted_combination_1990, p_combination_1990 = CW_test(actual_1990, y_pred_HA_1990, y_pred_combination_1990)
success_ratio_combination_1990, PT_combination_1990, p2_combination_1990 = PT_test(actual_1990, y_pred_combination_1990)
# success ratio of HA
success_ratio_HA_1990, PT_HA_1990, p2_HA_1990 = PT_test(actual_1990, y_pred_HA_1990)

/Users/xingfuxu/PycharmProjects/EquityPremiumPrediction3/Perform_PT_test.py:36: RuntimeWarning: invalid value encountered in double_scalars
  stat = (p_hat - p_star) / np.sqrt(p_hat_var - p_star_var)


In [16]:

# output results
results_OOS_sample_forecast1 = np.array([
    [np.nan, np.nan, np.nan, success_ratio_HA_1957, PT_HA_1957, p2_HA_1957,
     np.nan, np.nan, np.nan, success_ratio_HA_1990, PT_HA_1990, p2_HA_1990],
    [OOS_R_OLS_1957, MSFE_adjusted_OLS_1957, p_OLS_1957, success_ratio_OLS_1957, PT_OLS_1957, p2_OLS_1957, 
     OOS_R_OLS_1990, MSFE_adjusted_OLS_1990, p_OLS_1990, success_ratio_OLS_1990, PT_OLS_1990, p2_OLS_1990],
    [OOS_R_PLS_1957, MSFE_adjusted_PLS_1957, p_PLS_1957, success_ratio_PLS_1957, PT_PLS_1957, p2_PLS_1957,
     OOS_R_PLS_1990, MSFE_adjusted_PLS_1990, p_PLS_1990, success_ratio_PLS_1990, PT_PLS_1990, p2_PLS_1990],
    [OOS_R_PCR_1957, MSFE_adjusted_PCR_1957, p_PCR_1957, success_ratio_PCR_1957, PT_PCR_1957, p2_PCR_1957,
     OOS_R_PCR_1990, MSFE_adjusted_PCR_1990, p_PCR_1990, success_ratio_PCR_1990, PT_PCR_1990, p2_PCR_1990],
    [OOS_R_LASSO_1957, MSFE_adjusted_LASSO_1957, p_LASSO_1957, success_ratio_LASSO_1957, PT_LASSO_1957, p2_LASSO_1957,
     OOS_R_LASSO_1990, MSFE_adjusted_LASSO_1990, p_LASSO_1990, success_ratio_LASSO_1990, PT_LASSO_1990, p2_LASSO_1990],
    [OOS_R_ENet_1957, MSFE_adjusted_ENet_1957, p_ENet_1957, success_ratio_ENet_1957, PT_ENet_1957, p2_ENet_1957, 
     OOS_R_ENet_1990, MSFE_adjusted_ENet_1990, p_ENet_1990, success_ratio_ENet_1990, PT_ENet_1990, p2_ENet_1990],
    [OOS_R_GBRT_1957, MSFE_adjusted_GBRT_1957, p_GBRT_1957, success_ratio_GBRT_1957, PT_GBRT_1957, p2_GBRT_1957,
     OOS_R_GBRT_1990, MSFE_adjusted_GBRT_1990, p_GBRT_1990, success_ratio_GBRT_1990, PT_GBRT_1990, p2_GBRT_1990],
    [OOS_R_RF_1957, MSFE_adjusted_RF_1957, p_RF_1957, success_ratio_RF_1957, PT_RF_1957, p2_RF_1957, 
     OOS_R_RF_1990, MSFE_adjusted_RF_1990, p_RF_1990, success_ratio_RF_1990, PT_RF_1990, p2_RF_1990],
    [OOS_R_NN3_1957, MSFE_adjusted_NN3_1957, p_NN3_1957, success_ratio_NN3_1957, PT_NN3_1957, p2_NN3_1957,
     OOS_R_NN3_1990, MSFE_adjusted_NN3_1990, p_NN3_1990, success_ratio_NN3_1990, PT_NN3_1990, p2_NN3_1990]
])
results_OOS_sample_forecast1 = pd.DataFrame(results_OOS_sample_forecast1)
results_OOS_sample_forecast1.columns = ['1957-' + s for s in ['R2', 'CW_stat', 'pValue', 'Success_ratio', 'PT-stat', 'p2Value']] + \
    ['1990-' + s for s in ['R2', 'CW_stat', 'pValue', 'Success_ratio', 'PT-stat', 'p2Value']]
results_OOS_sample_forecast1.insert(0, "Forecasting models",  ["HA", "OLS", "PLS", "PCR", "LASSO", 
                                                               "ENet", "GBRT", "RF", "NN3"])
results_OOS_sample_forecast1.to_csv("results_OOS_sample_forecast1_quarterly.csv", index=False)
results_OOS_sample_forecast1

,Forecasting models,1957-R2,1957-CW_stat,1957-pValue,1957-Success_ratio,1957-PT-stat,1957-p2Value,1990-R2,1990-CW_stat,1990-pValue,1990-Success_ratio,1990-PT-stat,1990-p2Value
0,HA,NaN,NaN,NaN,0.650980,NaN,NaN,NaN,NaN,NaN,0.707317,NaN,NaN
1,OLS,-0.556980,-0.490746,0.688197,0.549020,0.644881,0.259502,-0.454000,-0.400646,0.655660,0.634146,1.533137,0.062621
2,PLS,-0.265339,-1.441054,0.925215,0.470588,-2.042125,0.979430,-0.408412,-2.419509,0.992229,0.463415,-1.525593,0.936444
3,PCR,-0.062125,-0.029615,0.511813,0.580392,0.110710,0.455923,-0.064136,0.130172,0.448215,0.634146,1.215454,0.112096
4,LASSO,-0.023064,-0.849461,0.802188,0.635294,0.065720,0.473800,-0.025887,-0.772989,0.780235,0.691057,0.531380,0.297578
5,ENet,-0.022469,-0.867392,0.807136,0.635294,0.065720,0.473800,-0.027781,-0.841496,0.799965,0.691057,0.531380,0.297578
6,GBRT,-0.463777,-0.312763,0.622770,0.556863,0.191111,0.424219,-0.809841,-1.125545,0.869821,0.569106,0.000000,0.500000
7,RF,-0.115431,-0.444531,0.671671,0.627451,0.281099,0.389317,-0.236527,-1.598860,0.945074,0.617886,-1.815371,0.965267
8,NN3,-0.007130,0.148085,0.441138,0.635294,-0.117010,0.546574,-0.071242,-2.732873,0.996861,0.650407,-1.248638,0.894101


In [17]:
results_OOS_sample_forecast2 = np.array([
    [np.nan, np.nan, np.nan, success_ratio_HA_1957, PT_HA_1957, p2_HA_1957,
     np.nan, np.nan, np.nan, success_ratio_HA_1990, PT_HA_1990, p2_HA_1990],
    [OOS_R_SVR_1957, MSFE_adjusted_SVR_1957, p_SVR_1957, success_ratio_SVR_1957, PT_SVR_1957, p2_SVR_1957,
     OOS_R_SVR_1990, MSFE_adjusted_SVR_1990, p_SVR_1990, success_ratio_SVR_1990, PT_SVR_1990, p2_SVR_1990],
    [OOS_R_KNR_1957, MSFE_adjusted_KNR_1957, p_KNR_1957, success_ratio_KNR_1957, PT_KNR_1957, p2_KNR_1957,
     OOS_R_KNR_1990, MSFE_adjusted_KNR_1990, p_KNR_1990, success_ratio_KNR_1990, PT_KNR_1990, p2_KNR_1990],
    [OOS_R_AdaBoost_1957, MSFE_adjusted_AdaBoost_1957, p_AdaBoost_1957, success_ratio_AdaBoost_1957, PT_AdaBoost_1957, p2_AdaBoost_1957,
     OOS_R_AdaBoost_1990, MSFE_adjusted_AdaBoost_1990, p_AdaBoost_1990, success_ratio_AdaBoost_1990, PT_AdaBoost_1990, p2_AdaBoost_1990],
    [OOS_R_XGBoost_1957, MSFE_adjusted_XGBoost_1957, p_XGBoost_1957, success_ratio_XGBoost_1957, PT_XGBoost_1957, p2_XGBoost_1957,
     OOS_R_XGBoost_1990, MSFE_adjusted_XGBoost_1990, p_XGBoost_1990, success_ratio_XGBoost_1990, PT_XGBoost_1990, p2_XGBoost_1990],
    [OOS_R_combination_1957, MSFE_adjusted_combination_1957, p_combination_1957, success_ratio_combination_1990, PT_combination_1990, p2_combination_1990,
     OOS_R_combination_1990, MSFE_adjusted_combination_1990, p_combination_1990, success_ratio_combination_1990, PT_combination_1990, p2_combination_1990]
])
#
results_OOS_sample_forecast2 = pd.DataFrame(results_OOS_sample_forecast2)
results_OOS_sample_forecast2.columns = ['1957-' + s for s in ['R2', 'CW_stat', 'pValue', 'Success_ratio', 'PT-stat', 'p2Value']] + \
    ['1990-' + s for s in ['R2', 'CW_stat', 'pValue', 'Success_ratio', 'PT-stat', 'p2Value']]
results_OOS_sample_forecast2.insert(0, "Forecasting models",
                                   ["HA", "SVR", "KNR", "AdaBoost", "XGBoost", "Combination"])
results_OOS_sample_forecast2.to_csv("results_OOS_sample_forecast2_quarterly.csv", index=False)
results_OOS_sample_forecast2

,Forecasting models,1957-R2,1957-CW_stat,1957-pValue,1957-Success_ratio,1957-PT-stat,1957-p2Value,1990-R2,1990-CW_stat,1990-pValue,1990-Success_ratio,1990-PT-stat,1990-p2Value
0,HA,NaN,NaN,NaN,0.650980,NaN,NaN,NaN,NaN,NaN,0.707317,NaN,NaN
1,SVR,-0.341742,0.221747,0.412256,0.541176,1.007363,0.156880,-0.516853,-0.506721,0.693825,0.593496,1.522777,0.063907
2,KNR,-0.063979,0.202893,0.419610,0.600000,0.665393,0.252899,-0.141605,-1.523377,0.936168,0.609756,-0.999575,0.841242
3,AdaBoost,-0.200252,-0.151856,0.560350,0.525490,0.081489,0.467527,-0.367020,-1.098546,0.864017,0.479675,0.039044,0.484427
4,XGBoost,-0.416049,0.577039,0.281957,0.529412,-0.067551,0.526929,-0.639237,-0.509678,0.694862,0.520325,-0.545236,0.707204
5,Combination,-0.077780,-0.264850,0.604438,0.601626,0.677918,0.248912,-0.182654,-1.171725,0.879346,0.601626,0.677918,0.248912
